# D180 - MySQL `CASE` / `WHEN`: Complete Practical Guide

`CASE` is a SQL **expression** that returns one value. It is used to derive labels, classify data, handle exceptions, build conditional aggregates, create pivot-style reports, define custom sorting, and apply conditional updates.

This notebook is self-contained. It connects to the running MySQL server using `root` / `root`, creates a small `case_when_lab` database, loads reusable sample data, and runs every example.

> Run the cells from top to bottom. The setup recreates only the two lab tables inside `case_when_lab`.

## 1. Two forms of `CASE`

### Simple `CASE`

Compares one expression with equality values:

```sql
CASE expression
    WHEN value_1 THEN result_1
    WHEN value_2 THEN result_2
    ELSE default_result
END
```

### Searched `CASE`

Evaluates independent Boolean conditions. Use it for ranges, `NULL`, `AND`, `OR`, dates, and comparisons between columns:

```sql
CASE
    WHEN condition_1 THEN result_1
    WHEN condition_2 THEN result_2
    ELSE default_result
END
```

MySQL evaluates conditions from top to bottom and returns the result for the **first true match**. If nothing matches and `ELSE` is omitted, the result is `NULL`.

## 2. Connect to MySQL

The connection defaults to 127.0.0.1:3306, username oot, and password oot. Environment variables can override these values. This works when the WSL MySQL port is reachable from Windows.

In [ ]:
import os
import mysql.connector
from mysql.connector import Error

MYSQL_CONFIG = {
    "host": os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    "port": int(os.environ.get("MYSQL_PORT", "3306")),
    "user": os.environ.get("MYSQL_USERNAME", "root"),
    "password": os.environ.get("MYSQL_PASSWORD", "root"),
}

server_connection = mysql.connector.connect(**MYSQL_CONFIG)
server_cursor = server_connection.cursor()
server_cursor.execute("CREATE DATABASE IF NOT EXISTS case_when_lab")
server_cursor.close()
server_connection.close()

connection = mysql.connector.connect(**MYSQL_CONFIG, database="case_when_lab")
print("Connected:", connection.is_connected())
print("MySQL version:", connection.get_server_info())

## 3. Reliable query helpers from D14/D16

print_rows prints aligned full result sets and displays SQL NULL clearly. execute_sql supports parameters, returns selected rows, commits successful DDL/DML, rolls back errors, and always closes its cursor.

In [ ]:
def print_rows(columns, rows):
    """Print a complete result set as an aligned text table."""
    if not rows:
        print("No rows returned.")
        return

    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None):
    """Execute one SQL statement; print and return rows or affected-row count."""
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [item[0] for item in cursor.description]
            rows = cursor.fetchall()
            print_rows(columns, rows)
            return rows

        affected_rows = cursor.rowcount
        connection.commit()
        print(f"Statement completed. Affected rows: {affected_rows:,}")
        return affected_rows
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## 4. Create reusable sample data

The tables deliberately include multiple statuses, boundary values, discounts, inactive customers, and NULL values. That makes the behavior of each pattern visible.

In [ ]:
setup_statements = [
    "DROP TABLE IF EXISTS sales",
    "DROP TABLE IF EXISTS customers",
    """
    CREATE TABLE customers (
        customer_id INT PRIMARY KEY,
        customer_name VARCHAR(50) NOT NULL,
        state_code CHAR(2) NOT NULL,
        signup_date DATE NOT NULL,
        credit_score INT NULL,
        is_active BOOLEAN NOT NULL
    )
    """,
    """
    CREATE TABLE sales (
        sale_id INT PRIMARY KEY,
        customer_id INT NOT NULL,
        order_date DATE NOT NULL,
        status VARCHAR(20) NOT NULL,
        channel VARCHAR(20) NOT NULL,
        amount DECIMAL(10,2) NOT NULL,
        discount DECIMAL(10,2) NULL,
        promised_date DATE NOT NULL,
        delivered_date DATE NULL,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    )
    """
]

for statement in setup_statements:
    execute_sql(statement)

customers = [
    (1, "Asha",   "KA", "2024-01-15", 810, True),
    (2, "Bala",   "TN", "2024-06-01", 690, True),
    (3, "Chitra", "KL", "2025-02-10", None, True),
    (4, "Deepak", "MH", "2023-11-20", 580, False),
    (5, "Esha",   "KA", "2025-08-05", 740, True),
]

sales = [
    (101, 1, "2025-07-01", "delivered", "web",   1200.00, 100.00, "2025-07-05", "2025-07-04"),
    (102, 1, "2025-07-10", "shipped",   "app",    450.00, None,   "2025-07-15", None),
    (103, 2, "2025-07-12", "delivered", "store",   80.00,   0.00, "2025-07-14", "2025-07-16"),
    (104, 2, "2025-08-01", "cancelled", "web",    250.00,  25.00, "2025-08-06", None),
    (105, 3, "2025-08-03", "pending",   "app",     50.00, None,   "2025-08-10", None),
    (106, 3, "2025-08-04", "delivered", "web",    500.00,  50.00, "2025-08-09", "2025-08-09"),
    (107, 4, "2025-08-05", "returned",  "store",  999.00, 100.00, "2025-08-08", "2025-08-07"),
    (108, 5, "2025-08-06", "delivered", "app",   1500.00, 200.00, "2025-08-12", "2025-08-11"),
    (109, 5, "2025-08-07", "processing","web",   300.00, None,   "2025-08-14", None),
]

customer_insert = """
INSERT INTO customers
(customer_id, customer_name, state_code, signup_date, credit_score, is_active)
VALUES (%s, %s, %s, %s, %s, %s)
"""
sale_insert = """
INSERT INTO sales
(sale_id, customer_id, order_date, status, channel, amount, discount,
 promised_date, delivered_date)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor = connection.cursor()
try:
    cursor.executemany(customer_insert, customers)
    cursor.executemany(sale_insert, sales)
    connection.commit()
    print(f"Loaded {len(customers)} customers and {len(sales)} sales.")
except Error:
    connection.rollback()
    raise
finally:
    cursor.close()

In [ ]:
execute_sql("SELECT * FROM sales ORDER BY sale_id")

## 5. Pattern: translate stored codes with simple CASE

Simple CASE is clearest when one column is compared with exact values. ELSE protects the report when a new or unexpected value appears.

In [ ]:
execute_sql("""
SELECT sale_id, status,
       CASE status
           WHEN 'pending'    THEN 'Waiting for review'
           WHEN 'processing' THEN 'Being prepared'
           WHEN 'shipped'    THEN 'In transit'
           WHEN 'delivered'  THEN 'Completed'
           WHEN 'cancelled'  THEN 'Stopped'
           WHEN 'returned'   THEN 'Reversed'
           ELSE 'Unknown status'
       END AS status_label
FROM sales
ORDER BY sale_id
""")

## 6. Pattern: derive numeric bands with searched CASE

Put narrower/higher-priority conditions first. The first match wins, so descending thresholds are a natural pattern. Notice that boundary values such as 500 belong to the first matching condition.

In [ ]:
execute_sql("""
SELECT sale_id, amount,
       CASE
           WHEN amount >= 1000 THEN 'Large'
           WHEN amount >= 500  THEN 'Medium'
           WHEN amount >= 100  THEN 'Small'
           ELSE 'Micro'
       END AS order_size
FROM sales
ORDER BY amount DESC
""")

### Why condition order matters

The next query is intentionally wrong: every amount at least 100 matches the first branch, so the later >= 500 and >= 1000 branches can never run.

In [ ]:
execute_sql("""
SELECT sale_id, amount,
       CASE
           WHEN amount >= 100  THEN '100+'
           WHEN amount >= 500  THEN '500+'       -- unreachable
           WHEN amount >= 1000 THEN '1000+'      -- unreachable
           ELSE 'Below 100'
       END AS badly_ordered_label
FROM sales
ORDER BY amount DESC
""")

## 7. Pattern: combine conditions with AND, OR, and parentheses

Searched CASE can express business rules involving several columns. Parentheses make operator precedence explicit.

In [ ]:
execute_sql("""
SELECT sale_id, channel, amount, status,
       CASE
           WHEN status = 'delivered' AND amount >= 1000 THEN 'High-value success'
           WHEN status IN ('cancelled', 'returned')      THEN 'Revenue at risk'
           WHEN channel IN ('web', 'app') AND amount < 100 THEN 'Digital micro-order'
           ELSE 'Standard'
       END AS business_segment
FROM sales
ORDER BY sale_id
""")

## 8. Pattern: handle `NULL` explicitly

`NULL` is not equal to anything - not even another `NULL`. Use `IS NULL` / `IS NOT NULL`. In a simple `CASE`, `WHEN NULL` does not reliably match a null expression.

In [ ]:
execute_sql("""
SELECT customer_id, customer_name, credit_score,
       CASE
           WHEN credit_score IS NULL THEN 'Not scored'
           WHEN credit_score >= 750  THEN 'Excellent'
           WHEN credit_score >= 650  THEN 'Good'
           ELSE 'Needs review'
       END AS credit_band,
       CASE
           WHEN credit_score IS NULL THEN 0
           ELSE credit_score
       END AS score_for_display
FROM customers
ORDER BY customer_id
""")

`COALESCE(discount, 0)` is shorter when the only rule is to replace `NULL`. Use `CASE` when the replacement depends on a condition.

In [ ]:
execute_sql("""
SELECT sale_id, amount, discount,
       amount - COALESCE(discount, 0) AS net_amount,
       CASE
           WHEN discount IS NULL THEN 'No discount recorded'
           WHEN discount = 0     THEN 'Zero discount'
           ELSE 'Discount applied'
       END AS discount_state
FROM sales
ORDER BY sale_id
""")

## 9. Pattern: derive date and delivery labels

CASE is often used to compare actual dates with promised dates. Check NULL first because undelivered orders have no actual delivery date.

In [ ]:
execute_sql("""
SELECT sale_id, promised_date, delivered_date,
       CASE
           WHEN delivered_date IS NULL THEN 'Not delivered'
           WHEN delivered_date < promised_date THEN 'Early'
           WHEN delivered_date = promised_date THEN 'On time'
           ELSE 'Late'
       END AS delivery_performance,
       CASE
           WHEN delivered_date IS NULL THEN NULL
           ELSE DATEDIFF(delivered_date, promised_date)
       END AS days_from_promise
FROM sales
ORDER BY sale_id
""")

## 10. Pattern: create calculated values

Every THEN branch can return an expression. This example derives a fee rate and fee amount. Keep compatible result types across branches.

In [ ]:
execute_sql("""
SELECT sale_id, channel, amount,
       CASE
           WHEN channel = 'store' THEN 0.00
           WHEN amount >= 1000    THEN 0.01
           ELSE 0.02
       END AS fee_rate,
       ROUND(amount * CASE
           WHEN channel = 'store' THEN 0.00
           WHEN amount >= 1000    THEN 0.01
           ELSE 0.02
       END, 2) AS fee_amount
FROM sales
ORDER BY sale_id
""")

## 11. Pattern: protect calculations such as division

A searched CASE can avoid division by zero or return a business-specific result. NULLIF(denominator, 0) is a concise alternative when returning NULL is acceptable.

In [ ]:
execute_sql("""
SELECT sale_id, amount, COALESCE(discount, 0) AS discount,
       CASE
           WHEN amount = 0 THEN NULL
           ELSE ROUND(100 * COALESCE(discount, 0) / amount, 2)
       END AS discount_percent_case,
       ROUND(100 * COALESCE(discount, 0) / NULLIF(amount, 0), 2)
           AS discount_percent_nullif
FROM sales
ORDER BY sale_id
""")

## 12. Pattern: custom sorting with CASE in ORDER BY

Map business priorities to numbers, sort by that expression, and add stable tie-break columns. The sort expression does not have to appear in the output.

In [ ]:
execute_sql("""
SELECT sale_id, status, amount
FROM sales
ORDER BY CASE status
             WHEN 'pending'    THEN 1
             WHEN 'processing' THEN 2
             WHEN 'shipped'    THEN 3
             WHEN 'delivered'  THEN 4
             WHEN 'returned'   THEN 5
             WHEN 'cancelled'  THEN 6
             ELSE 7
         END,
         amount DESC,
         sale_id
""")

## 13. Pattern: conditional aggregation

CASE inside an aggregate is one of the most useful reporting patterns. It calculates several conditional metrics in one table scan.

- SUM(CASE WHEN ... THEN 1 ELSE 0 END) counts matching rows.
- SUM(CASE WHEN ... THEN amount ELSE 0 END) sums matching values.
- COUNT(CASE WHEN ... THEN 1 END) counts non-NULL matches.
- AVG(CASE WHEN ... THEN amount END) averages only matching rows because aggregates ignore NULL.

In [ ]:
execute_sql("""
SELECT channel,
       COUNT(*) AS all_orders,
       SUM(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
       COUNT(CASE WHEN status IN ('cancelled', 'returned') THEN 1 END) AS problem_orders,
       ROUND(SUM(CASE WHEN status = 'delivered' THEN amount ELSE 0 END), 2)
           AS delivered_value,
       ROUND(AVG(CASE WHEN status = 'delivered' THEN amount END), 2)
           AS average_delivered_value
FROM sales
GROUP BY channel
ORDER BY channel
""")

### A common counting mistake

COUNT(expression) counts every non-NULL expression. Therefore COUNT(CASE ... ELSE 0 END) counts all rows, because both 1 and 0 are non-NULL. Omit ELSE or return NULL for non-matches.

In [ ]:
execute_sql("""
SELECT
    COUNT(CASE WHEN status = 'delivered' THEN 1 END) AS correct_count,
    COUNT(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) AS wrong_count,
    SUM(CASE WHEN status = 'delivered' THEN 1 ELSE 0 END) AS correct_sum_pattern
FROM sales
""")

## 14. Pattern: pivot categories into columns

MySQL has no general PIVOT clause. Conditional aggregation creates compact cross-tab reports. Add one aggregate expression per required output column.

In [ ]:
execute_sql("""
SELECT channel,
       SUM(CASE WHEN status = 'delivered' THEN amount ELSE 0 END) AS delivered_value,
       SUM(CASE WHEN status = 'pending' THEN amount ELSE 0 END) AS pending_value,
       SUM(CASE WHEN status = 'processing' THEN amount ELSE 0 END) AS processing_value,
       SUM(CASE WHEN status = 'shipped' THEN amount ELSE 0 END) AS shipped_value,
       SUM(CASE WHEN status IN ('cancelled', 'returned') THEN amount ELSE 0 END)
           AS reversed_or_cancelled_value
FROM sales
GROUP BY channel
ORDER BY channel
""")

## 15. Pattern: group by a derived label

MySQL allows a select alias in GROUP BY, but repeating the expression is more portable across database systems. A CTE is often clearest when the expression is long.

In [ ]:
execute_sql("""
WITH labelled AS (
    SELECT sale_id, amount,
           CASE
               WHEN amount >= 1000 THEN 'Large'
               WHEN amount >= 500  THEN 'Medium'
               WHEN amount >= 100  THEN 'Small'
               ELSE 'Micro'
           END AS order_size
    FROM sales
)
SELECT order_size, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS total_amount
FROM labelled
GROUP BY order_size
ORDER BY MIN(amount)
""")

## 16. Pattern: filter aggregated results with HAVING

WHERE filters input rows before grouping. HAVING filters completed groups. CASE can build conditional group metrics used by HAVING.

In [ ]:
execute_sql("""
SELECT customer_id,
       COUNT(*) AS orders,
       SUM(CASE WHEN status = 'delivered' THEN amount ELSE 0 END) AS delivered_value
FROM sales
GROUP BY customer_id
HAVING SUM(CASE WHEN status = 'delivered' THEN amount ELSE 0 END) >= 500
ORDER BY delivered_value DESC
""")

## 17. Pattern: conditional ordering inside a window function

CASE can participate in a window's ORDER BY. Here completed sales are prioritized before other statuses, then rows are ordered by amount.

In [ ]:
execute_sql("""
SELECT sale_id, customer_id, status, amount,
       ROW_NUMBER() OVER (
           PARTITION BY customer_id
           ORDER BY CASE WHEN status = 'delivered' THEN 0 ELSE 1 END,
                    amount DESC,
                    sale_id
       ) AS customer_priority_row
FROM sales
ORDER BY customer_id, customer_priority_row
""")

## 18. Pattern: `CASE` in `WHERE` for optional parameters

This is possible, but direct Boolean predicates are usually clearer and can use indexes more effectively. Prefer `WHERE (%s IS NULL OR status = %s)` for optional parameters. The example uses safe parameter binding - never build user values into SQL strings.

In [ ]:
status_filter = "delivered"   # change to None to return every status

execute_sql("""
SELECT sale_id, status, amount
FROM sales
WHERE CASE
          WHEN %s IS NULL THEN TRUE
          WHEN status = %s THEN TRUE
          ELSE FALSE
      END
ORDER BY sale_id
""", (status_filter, status_filter))

### Preferred optional-filter form

In [ ]:
status_filter = None

execute_sql("""
SELECT sale_id, status, amount
FROM sales
WHERE (%s IS NULL OR status = %s)
ORDER BY sale_id
""", (status_filter, status_filter))

## 19. Pattern: conditional UPDATE

CASE can update different rows to different values in one statement. Always preview the logic with a SELECT, use a restrictive WHERE, and use a transaction for important data. This lab example changes only two known rows and is safe to rerun because it toggles between the same values.

In [ ]:
# Preview old and proposed values before updating.
execute_sql("""
SELECT sale_id, status AS old_status,
       CASE
           WHEN status = 'pending' THEN 'processing'
           WHEN status = 'processing' THEN 'pending'
           ELSE status
       END AS proposed_status
FROM sales
WHERE sale_id IN (105, 109)
ORDER BY sale_id
""")

In [ ]:
execute_sql("""
UPDATE sales
SET status = CASE
                 WHEN status = 'pending' THEN 'processing'
                 WHEN status = 'processing' THEN 'pending'
                 ELSE status
             END
WHERE sale_id IN (105, 109)
""")

execute_sql("SELECT sale_id, status FROM sales WHERE sale_id IN (105, 109) ORDER BY sale_id")

## 20. Pattern: nested CASE

Nested CASE is legal, but a flat searched CASE is often easier to maintain. Use nesting when the second decision genuinely belongs inside the first.

In [ ]:
execute_sql("""
SELECT customer_id, customer_name, is_active, credit_score,
       CASE
           WHEN is_active = FALSE THEN 'Inactive'
           ELSE CASE
                    WHEN credit_score IS NULL THEN 'Active / unscored'
                    WHEN credit_score >= 750 THEN 'Active / premium'
                    ELSE 'Active / standard'
                END
       END AS customer_segment
FROM customers
ORDER BY customer_id
""")

## 21. Result types and implicit conversion

MySQL finds a common result type across all THEN and ELSE branches. Mixing numbers, dates, and strings can cause surprising conversion. Prefer compatible branch types and use CAST explicitly when needed.

In [ ]:
execute_sql("""
SELECT sale_id, amount,
       CASE
           WHEN amount >= 1000 THEN CAST('1000.00' AS DECIMAL(10,2))
           ELSE CAST(amount AS DECIMAL(10,2))
       END AS capped_amount
FROM sales
ORDER BY sale_id
""")

## 22. `CASE` versus related MySQL functions

| Requirement | Best fit |
|---|---|
| Several portable conditions | `CASE` |
| Two outcomes in MySQL-only code | `IF(condition, true_value, false_value)` |
| First non-null value | `COALESCE(a, b, c)` |
| Replace one exact value with `NULL` | `NULLIF(a, value)` |
| Stored-program control flow | `IF ... THEN`, not the `CASE` expression |

`CASE` is standard SQL and usually the best default for readable, portable query logic.

In [ ]:
execute_sql("""
SELECT sale_id, discount,
       CASE WHEN discount IS NULL THEN 0 ELSE discount END AS via_case,
       IFNULL(discount, 0) AS via_ifnull,
       COALESCE(discount, 0) AS via_coalesce
FROM sales
ORDER BY sale_id
""")

## 23. Practical rules and pitfalls

1. `CASE` returns a value; it is not a loop or a stored-program control-flow block.
2. Conditions are checked top to bottom; the first true branch wins.
3. Put special cases and narrow rules before broad rules.
4. Include `ELSE` unless returning `NULL` is intentional.
5. Use `IS NULL`, never `= NULL`.
6. Keep branch result types compatible; cast explicitly when necessary.
7. For counts, use `SUM(CASE ... THEN 1 ELSE 0 END)` or `COUNT(CASE ... THEN 1 END)`.
8. Do not use `COUNT(CASE ... ELSE 0 END)` to count matches.
9. Prefer direct predicates in `WHERE`; wrapping indexed columns in expressions can hurt index use.
10. Do not repeat very large `CASE` expressions everywhere. Calculate once in a CTE, view, or maintained dimension table.
11. A large code-to-label mapping often belongs in a lookup table, not a giant `CASE`.
12. Test boundaries, overlaps, unexpected values, and `NULL` inputs.

## 24. Use-case checklist

- derive display labels and business segments;
- bucket numeric values and dates;
- normalize codes or statuses;
- handle missing values conditionally;
- calculate conditional measures;
- conditional aggregation and row-to-column reports;
- custom business sorting;
- conditional updates;
- priority rules inside window functions;
- safe arithmetic and exception handling;
- parameter-driven logic.

## 25. Short practice tasks

1. Label each customer as New or Established using signup_date.
2. Create 
et_amount bands after subtracting discount.
3. Count on-time and late deliveries per channel in one query.
4. Sort statuses as pending, processing, shipped, delivered, returned, cancelled.
5. Return each customer's highest-priority order using ROW_NUMBER and CASE.

## 26. Close the connection

In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")